# Spark Structured Streaming - Real-time Data Processing

This notebook demonstrates Apache Spark's Structured Streaming capabilities for real-time data processing:
- Reading streaming data from files
- Stream processing and transformations
- Windowing and aggregations
- Stream-to-stream joins
- Output modes and triggers
- State management and checkpointing

## Setup and Configuration

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import json
import time
import threading
import random
from datetime import datetime, timedelta

# Create Spark Session with streaming optimizations
spark = SparkSession.builder \
    .appName("Structured Streaming Demo") \
    .master("spark://spark-master:7077") \
    .config("spark.sql.streaming.checkpointLocation", "/home/jovyan/data/checkpoints") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.streaming.forceDeleteTempCheckpointLocation", "true") \
    .getOrCreate()

# Set log level to reduce output
spark.sparkContext.setLogLevel("WARN")

print(f"Spark Version: {spark.version}")
print("Streaming demo environment ready!")

## 1. Create Streaming Data Generator

In [ ]:
import os

# Create directories for streaming data
os.makedirs('/home/jovyan/data/streaming/input', exist_ok=True)
os.makedirs('/home/jovyan/data/streaming/output', exist_ok=True)
os.makedirs('/home/jovyan/data/checkpoints', exist_ok=True)

# Function to generate streaming web events
def generate_web_events():
    """Generate simulated web events for streaming"""
    events = ['page_view', 'click', 'purchase', 'signup', 'logout', 'add_to_cart']
    devices = ['desktop', 'mobile', 'tablet']
    browsers = ['chrome', 'firefox', 'safari', 'edge']
    pages = ['/home', '/products', '/cart', '/checkout', '/profile', '/search']
    
    file_counter = 0
    
    while file_counter < 5:  # Generate 5 files for demo
        events_batch = []
        
        # Generate 100 events per batch
        for i in range(100):
            event = {
                'timestamp': datetime.now().isoformat(),
                'user_id': f'USER{random.randint(1, 1000):04d}',
                'session_id': f'SESSION{random.randint(1, 10000):05d}',
                'event_type': random.choice(events),
                'device': random.choice(devices),
                'browser': random.choice(browsers),
                'page_url': random.choice(pages),
                'duration_seconds': random.randint(1, 300),
                'referrer': random.choice(['google', 'facebook', 'direct', 'twitter', '']),
                'ip_address': f'{random.randint(1,255)}.{random.randint(1,255)}.{random.randint(1,255)}.{random.randint(1,255)}',
                'amount': round(random.uniform(0, 500), 2) if random.choice(events) == 'purchase' else None
            }
            events_batch.append(event)
        
        # Write batch to file
        filename = f'/home/jovyan/data/streaming/input/events_{file_counter:03d}.json'
        with open(filename, 'w') as f:
            for event in events_batch:
                f.write(json.dumps(event) + '\n')
        
        print(f"Generated {filename} with {len(events_batch)} events")
        file_counter += 1
        time.sleep(10)  # Wait 10 seconds between files

# Start the generator in a separate thread (for demo purposes)
print("Starting event generation...")
generator_thread = threading.Thread(target=generate_web_events)
generator_thread.daemon = True
generator_thread.start()

# Wait a moment for first file to be generated
time.sleep(12)
print("Event generation started!")

## 2. Define Schema for Streaming Data

In [ ]:
# Define schema for web events
web_events_schema = StructType([
    StructField("timestamp", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("session_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("device", StringType(), True),
    StructField("browser", StringType(), True),
    StructField("page_url", StringType(), True),
    StructField("duration_seconds", IntegerType(), True),
    StructField("referrer", StringType(), True),
    StructField("ip_address", StringType(), True),
    StructField("amount", DoubleType(), True)
])

print("Schema defined for streaming data")
print(web_events_schema)

## 3. Create Streaming DataFrame

In [ ]:
# Create streaming DataFrame
streaming_df = spark.readStream \
    .format("json") \
    .schema(web_events_schema) \
    .option("path", "/home/jovyan/data/streaming/input") \
    .option("maxFilesPerTrigger", 1) \
    .load()

# Add processed timestamp and convert string timestamp to TimestampType
streaming_df = streaming_df \
    .withColumn("event_timestamp", to_timestamp(col("timestamp"))) \
    .withColumn("processed_time", current_timestamp()) \
    .drop("timestamp")

print("Streaming DataFrame created")
print(f"Is streaming: {streaming_df.isStreaming}")
streaming_df.printSchema()

## 4. Basic Stream Processing - Real-time Event Count

In [ ]:
# Simple aggregation: count events by type
event_counts = streaming_df \
    .groupBy("event_type") \
    .count() \
    .orderBy(desc("count"))

# Start the streaming query
query1 = event_counts.writeStream \
    .outputMode("complete") \
    .format("console") \
    .option("truncate", "false") \
    .trigger(processingTime='10 seconds') \
    .start()

print("Started real-time event counting...")
print("Query ID:", query1.id)

# Let it run for a bit
time.sleep(25)
query1.stop()
print("Stopped event counting query")

## 5. Windowed Aggregations - Time-based Analytics

In [ ]:
# Windowed aggregations: events per minute
windowed_counts = streaming_df \
    .withWatermark("event_timestamp", "1 minute") \
    .groupBy(
        window(col("event_timestamp"), "1 minute"),
        "event_type"
    ) \
    .count() \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        "event_type",
        "count"
    ) \
    .orderBy("window_start", desc("count"))

# Start windowed query
query2 = windowed_counts.writeStream \
    .outputMode("update") \
    .format("console") \
    .option("truncate", "false") \
    .trigger(processingTime='10 seconds') \
    .start()

print("Started windowed aggregation...")
time.sleep(25)
query2.stop()
print("Stopped windowed query")

## 6. Complex Stream Processing - User Session Analytics

In [ ]:
# More complex processing: user session metrics
session_metrics = streaming_df \
    .withWatermark("event_timestamp", "2 minutes") \
    .groupBy(
        window(col("event_timestamp"), "2 minutes"),
        "user_id",
        "session_id"
    ) \
    .agg(
        count("*").alias("events_in_session"),
        countDistinct("page_url").alias("unique_pages"),
        sum("duration_seconds").alias("total_duration"),
        max("event_timestamp").alias("last_event"),
        min("event_timestamp").alias("first_event"),
        sum(when(col("event_type") == "purchase", col("amount")).otherwise(0)).alias("total_purchase_amount")
    ) \
    .select(
        col("window.start").alias("window_start"),
        "user_id",
        "session_id",
        "events_in_session",
        "unique_pages",
        "total_duration",
        "total_purchase_amount"
    ) \
    .filter(col("events_in_session") > 1)  # Only sessions with multiple events

# Write to memory table for analysis
query3 = session_metrics.writeStream \
    .outputMode("update") \
    .format("memory") \
    .queryName("session_metrics") \
    .trigger(processingTime='10 seconds') \
    .start()

print("Started session metrics processing...")
time.sleep(30)

# Query the in-memory table
spark.sql("SELECT * FROM session_metrics ORDER BY total_purchase_amount DESC").show(10)

query3.stop()
print("Stopped session metrics query")

## 7. Stream-to-Static Join

In [ ]:
# Load static customer data for enrichment
customers_static = spark.read.csv("/home/jovyan/data/csv/customers.csv", header=True, inferSchema=True) \
    .select("customer_id", "first_name", "last_name", "age", "city", "loyalty_tier")

# Create a mapping from user_id to customer_id (for demo purposes)
# In real scenarios, you'd have a proper user-to-customer mapping
user_customer_mapping = spark.range(1, 1001) \
    .withColumn("user_id", concat(lit("USER"), format_string("%04d", col("id")))) \
    .withColumn("customer_id", concat(lit("CUST"), format_string("%04d", col("id")))) \
    .select("user_id", "customer_id")

# Enrich streaming data with customer information
enriched_stream = streaming_df \
    .join(user_customer_mapping, "user_id", "left") \
    .join(customers_static, "customer_id", "left") \
    .select(
        "event_timestamp",
        "user_id",
        "customer_id",
        "first_name",
        "last_name",
        "loyalty_tier",
        "event_type",
        "device",
        "amount"
    )

# Analyze purchases by loyalty tier
loyalty_purchases = enriched_stream \
    .filter(col("event_type") == "purchase") \
    .filter(col("amount").isNotNull()) \
    .groupBy("loyalty_tier") \
    .agg(
        count("*").alias("purchase_count"),
        round(sum("amount"), 2).alias("total_amount"),
        round(avg("amount"), 2).alias("avg_amount")
    )

query4 = loyalty_purchases.writeStream \
    .outputMode("complete") \
    .format("console") \
    .trigger(processingTime='15 seconds') \
    .start()

print("Started loyalty tier analysis...")
time.sleep(30)
query4.stop()
print("Stopped loyalty tier analysis")

## 8. Writing Streams to Different Sinks

In [ ]:
# Prepare different output streams

# 1. Write raw events to Parquet files (append mode)
parquet_query = streaming_df \
    .select("event_timestamp", "user_id", "event_type", "device", "page_url", "amount") \
    .writeStream \
    .format("parquet") \
    .outputMode("append") \
    .option("path", "/home/jovyan/data/streaming/output/parquet") \
    .option("checkpointLocation", "/home/jovyan/data/checkpoints/parquet") \
    .trigger(processingTime='20 seconds') \
    .start()

# 2. Write aggregated data to JSON files
json_query = streaming_df \
    .groupBy("device", "event_type") \
    .count() \
    .writeStream \
    .format("json") \
    .outputMode("complete") \
    .option("path", "/home/jovyan/data/streaming/output/json") \
    .option("checkpointLocation", "/home/jovyan/data/checkpoints/json") \
    .trigger(processingTime='20 seconds') \
    .start()

print("Started output streams...")
print(f"Parquet query ID: {parquet_query.id}")
print(f"JSON query ID: {json_query.id}")

# Monitor for a while
time.sleep(45)

# Stop queries
parquet_query.stop()
json_query.stop()
print("Stopped output streams")

## 9. Stream Monitoring and Management

In [ ]:
# Check what files were created
import os

def list_files_recursive(directory):
    for root, dirs, files in os.walk(directory):
        level = root.replace(directory, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        sub_indent = ' ' * 2 * (level + 1)
        for file in files:
            print(f"{sub_indent}{file}")

print("Generated output files:")
list_files_recursive("/home/jovyan/data/streaming/output")

print("\nCheckpoint directories:")
list_files_recursive("/home/jovyan/data/checkpoints")

In [ ]:
# Read back the saved Parquet data to verify
saved_events = spark.read.parquet("/home/jovyan/data/streaming/output/parquet")
print(f"Total events saved: {saved_events.count()}")
print("Sample of saved events:")
saved_events.show(10)

# Analyze the saved data
print("\nEvent distribution in saved data:")
saved_events.groupBy("event_type").count().orderBy(desc("count")).show()

## 10. Advanced Streaming Patterns

In [ ]:
# Deduplication in streaming
deduplicated_stream = streaming_df \
    .withWatermark("event_timestamp", "1 minute") \
    .dropDuplicates(["user_id", "session_id", "event_timestamp"])

# Complex event processing: detect user journey patterns
user_journey = streaming_df \
    .withWatermark("event_timestamp", "5 minutes") \
    .groupBy(
        window(col("event_timestamp"), "5 minutes"),
        "user_id",
        "session_id"
    ) \
    .agg(
        collect_list("event_type").alias("event_sequence"),
        collect_list("page_url").alias("page_sequence"),
        count("*").alias("total_events"),
        max("event_timestamp").alias("session_end"),
        min("event_timestamp").alias("session_start")
    ) \
    .withColumn("session_duration_minutes", 
                (col("session_end").cast("long") - col("session_start").cast("long")) / 60) \
    .withColumn("has_purchase", array_contains(col("event_sequence"), "purchase")) \
    .withColumn("conversion_funnel", 
                when(array_contains(col("event_sequence"), "purchase"), "Converted")
                .when(array_contains(col("event_sequence"), "add_to_cart"), "Added to Cart")
                .when(array_contains(col("event_sequence"), "click"), "Engaged")
                .otherwise("Visitor"))

# Monitor user journeys
journey_query = user_journey \
    .filter(col("total_events") > 2) \
    .writeStream \
    .outputMode("update") \
    .format("console") \
    .option("truncate", "false") \
    .trigger(processingTime='15 seconds') \
    .start()

print("Started user journey analysis...")
time.sleep(30)
journey_query.stop()
print("Stopped user journey analysis")

## 11. Cleanup and Best Practices

In [ ]:
# Function to monitor all active streams
def monitor_active_streams():
    active_streams = spark.streams.active
    print(f"Active streams: {len(active_streams)}")
    
    for stream in active_streams:
        print(f"  Stream ID: {stream.id}")
        print(f"  Name: {stream.name}")
        print(f"  Status: {stream.isActive}")
        
        # Get recent progress
        progress = stream.recentProgress
        if progress:
            latest = progress[-1]
            print(f"  Batch ID: {latest.get('batchId', 'N/A')}")
            print(f"  Input rows: {latest.get('inputRowsPerSecond', 'N/A')}")
            print(f"  Processing time: {latest.get('durationMs', {}).get('triggerExecution', 'N/A')} ms")
        print()

monitor_active_streams()

# Stop any remaining active streams
for stream in spark.streams.active:
    print(f"Stopping stream: {stream.id}")
    stream.stop()

print("All streams stopped")

## Summary

This notebook demonstrated comprehensive Spark Structured Streaming capabilities:

### Key Concepts Covered:
1. **Streaming DataFrames**: Reading from file sources with schema inference
2. **Basic Aggregations**: Real-time event counting and grouping
3. **Windowed Operations**: Time-based analytics with watermarking
4. **Complex Processing**: Session analysis and user journey tracking
5. **Stream Enrichment**: Joining streaming data with static datasets
6. **Multiple Output Modes**: Complete, Update, and Append modes
7. **Various Sinks**: Console, Memory, Parquet, and JSON outputs
8. **Advanced Patterns**: Deduplication and complex event processing
9. **Monitoring**: Stream management and progress tracking

### Best Practices Demonstrated:
- **Schema Definition**: Always define schemas for better performance
- **Watermarking**: Handle late-arriving data appropriately
- **Checkpointing**: Ensure fault tolerance and exactly-once processing
- **Resource Management**: Monitor and stop streams properly
- **Output Modes**: Choose appropriate modes for different use cases

### Real-world Applications:
- Real-time analytics dashboards
- Fraud detection systems
- IoT data processing
- User behavior analysis
- System monitoring and alerting

### Next Steps:
- Explore Kafka integration for production streaming
- Implement machine learning on streaming data
- Set up monitoring and alerting systems
- Scale streaming applications across clusters